# 🧠 딥러닝(PyTorch) 한 바퀴 둘러보기
텐서의 **모양(shape)** 이 층을 지날 때마다 어떻게 바뀌는지가 핵심입니다. 변수 칸에서 모양과 그림을 같이 확인하세요.

(필요: `pip install torch`)

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. 가짜 이미지 배치 만들기 — (개수, 색, 높이, 너비)

In [ ]:
yy, xx = torch.meshgrid(torch.arange(16), torch.arange(16), indexing="ij")
images = torch.stack([
    torch.stack([((xx - 8) ** 2 + (yy - 8) ** 2 < r ** 2).float()] * 3) * torch.rand(3, 1, 1)
    for r in range(2, 8)
])
labels = torch.tensor([0, 0, 0, 1, 1, 1])     # 작은 원 = 0, 큰 원 = 1
images.shape

## 2. 작은 CNN 모델

In [ ]:
model = nn.Sequential(
    nn.Conv2d(3, 4, kernel_size=3, padding=1),   # 필터 4개
    nn.ReLU(),
    nn.MaxPool2d(2),                              # 크기 절반
    nn.Flatten(),
    nn.Linear(4 * 8 * 8, 2),
)
model

## 3. 층을 하나씩 통과시키며 모양 보기

In [ ]:
features = model[0](images)      # 특징 맵 (6, 4, 16, 16)
activated = model[1](features)   # 음수는 0으로
pooled = model[2](activated)     # (6, 4, 8, 8)
flat = model[3](pooled)          # (6, 256)
logits = model[4](flat)          # (6, 2)

## 4. 필터(가중치)는 3×3 숫자들

In [ ]:
filters = model[0].weight
filters.shape

## 5. 몇 번 학습해 보기 — loss 가 줄어드는지 확인

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
losses = []
for epoch in range(5):
    logits = model(images)
    loss = nn.functional.cross_entropy(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(round(loss.item(), 4))
losses

In [ ]:
pred = model(images).argmax(dim=1)
accuracy = (pred == labels).float().mean().item()
accuracy